In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# This is the LangChain way of creating an LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile"
)

print("✅ LangChain + Groq ready!")

✅ LangChain + Groq ready!


In [4]:
# Step 1 - Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains things simply."),
    ("user", "{topic} - explain this in 3 sentences for a beginner.")
])

# Step 2 - Create output parser
parser = StrOutputParser()

# Step 3 - Chain them together with | pipe
chain = prompt | llm | parser

# Step 4 - Run it!
result = chain.invoke({"topic": "Machine Learning"})
print(result)

Machine learning is a way for computers to learn and improve on their own by using data, rather than being programmed with specific instructions. It's like teaching a child to recognize pictures of animals - you show them many examples, and they start to figure out what makes a cat or dog, so they can recognize new ones they've never seen before. By using machine learning, computers can do things like recognize images, understand speech, and make predictions, all without being explicitly told how to do it.


In [5]:
# Same chain, different topics - no rewriting!
topics = [
    "Neural Networks",
    "Blockchain",
    "Quantum Computing"
]

for topic in topics:
    result = chain.invoke({"topic": topic})
    print(f"\n📚 {topic}:")
    print(result)
    print("-" * 50)


📚 Neural Networks:
A neural network is a type of computer system that is designed to think and learn like the human brain, made up of many connected "neurons" that process and share information. These networks can be trained on large amounts of data, such as images or sounds, to recognize patterns and make predictions or decisions. By learning from the data, neural networks can become very good at tasks like recognizing pictures, understanding speech, or making recommendations, and are used in many applications like self-driving cars, virtual assistants, and social media.
--------------------------------------------------

📚 Blockchain:
A blockchain is a digital record book that stores information in a way that's secure, transparent, and can't be altered. It's like a long list of transactions, where each "block" of information is linked to the previous one, creating a "chain" that shows everything that's happened. This technology allows people to trust and verify the information on th

In [6]:
# Prompt with multiple variables
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher who adapts explanations for different audiences."),
    ("user", "Explain {topic} for {audience}. Keep it under 100 words.")
])

chain2 = prompt2 | llm | parser

# Same topic, different audiences
audiences = [
    ("Artificial Intelligence", "a 10 year old child"),
    ("Artificial Intelligence", "a business executive"),
    ("Artificial Intelligence", "a software engineer"),
]

for topic, audience in audiences:
    result = chain2.invoke({"topic": topic, "audience": audience})
    print(f"\n🎯 For {audience}:")
    print(result)
    print("-" * 50)


🎯 For a 10 year old child:
Artificial Intelligence (AI) is like a super smart computer that can learn and do things on its own. It's like a robot that can play games, recognize pictures, and even talk to you. AI helps us with many things, like self-driving cars and virtual assistants like Siri or Alexa. It's like having a magic helper that makes our lives easier and more fun!
--------------------------------------------------

🎯 For a business executive:
Artificial Intelligence (AI) refers to computer systems that can think and learn like humans. In business, AI can analyze data, automate tasks, and make informed decisions, boosting efficiency and innovation. It can be applied to areas like customer service, marketing, and operations, helping you stay competitive and drive growth. Think of AI as a smart tool that enhances human capabilities, freeing up time for strategic decision-making and driving business success.
--------------------------------------------------

🎯 For a software 

In [7]:
# Streaming - see words appear one by one
prompt3 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{question}")
])

chain3 = prompt3 | llm | parser

print("Streaming response:\n")

# stream() instead of invoke()
for chunk in chain3.stream({"question": "Tell me 5 interesting facts about space"}):
    print(chunk, end="", flush=True)

print("\n\nDone!")

Streaming response:

Here are 5 interesting facts about space:

1. **The Andromeda Galaxy is Coming for Us**: The Andromeda Galaxy, our closest galactic neighbor, is approaching the Milky Way at a speed of about 250,000 miles per hour (400,000 km/h). In about 4.5 billion years, the two galaxies will collide, resulting in a spectacular cosmic merger. Don't worry, though - the distances between stars are so vast that the chances of any stars or planets actually colliding are extremely low.

2. **Space is Filled with Mysterious Fast Radio Bursts**: Fast Radio Bursts (FRBs) are brief, intense pulses of energy that originate from distant galaxies. These enigmatic events were first discovered in 2007, and scientists are still trying to understand what causes them. Some theories suggest that FRBs could be the result of massive supernovae or the collapse of massive stars, while others propose that they might be evidence of advanced alien technology.

3. **The International Space Station is Hug

In [8]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel
from typing import List

# Define the STRUCTURE we want back
class Research(BaseModel):
    summary: str
    key_facts: List[str]
    subtopics: List[str]
    difficulty: str

# Create JSON parser
json_parser = JsonOutputParser(pydantic_object=Research)

# Prompt that forces JSON output
research_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research expert. 
Always respond with valid JSON only. No extra text.
Your response must have these exact fields:
- summary: a 2-3 sentence explanation
- key_facts: list of exactly 5 interesting facts
- subtopics: list of exactly 3 related topics to explore
- difficulty: one word - either beginner, intermediate, or advanced"""),
    ("user", "Research this topic thoroughly: {topic}")
])

# Build the chain
research_chain = research_prompt | llm | json_parser

# Test it!
result = research_chain.invoke({"topic": "Machine Learning"})

print("📝 SUMMARY:")
print(result["summary"])

print("\n🔑 KEY FACTS:")
for i, fact in enumerate(result["key_facts"], 1):
    print(f"{i}. {fact}")

print("\n🎯 SUBTOPICS TO EXPLORE:")
for topic in result["subtopics"]:
    print(f"• {topic}")

print("\n📊 DIFFICULTY:", result["difficulty"])

📝 SUMMARY:
Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve their performance over time. It has numerous applications in areas such as computer vision, natural language processing, and predictive analytics. Machine learning algorithms can be categorized into supervised, unsupervised, and reinforcement learning.

🔑 KEY FACTS:
1. Machine learning is a key driver of artificial intelligence and has numerous applications in industries such as healthcare, finance, and transportation
2. Deep learning is a type of machine learning that uses neural networks with multiple layers to analyze data
3. Machine learning algorithms can be trained on large datasets to improve their accuracy and performance
4. Unsupervised machine learning involves training models on unlabeled data to discover patterns and relationships
5. Reinforcement learning involves training models to make decisions based on rewards or penalties

🎯 SUBTOPICS TO EXPLORE:
• Ne

In [9]:
# Single topic version - works better in Jupyter
topic = "Artificial Intelligence"  # Change this to any topic you want!

print(f"⏳ Researching {topic}...")

result = research_chain.invoke({"topic": topic})

print("\n📝 SUMMARY:")
print(result["summary"])

print("\n🔑 KEY FACTS:")
for i, fact in enumerate(result["key_facts"], 1):
    print(f"{i}. {fact}")

print("\n🎯 SUBTOPICS TO EXPLORE:")
for subtopic in result["subtopics"]:
    print(f"• {subtopic}")

print("\n📊 DIFFICULTY:", result["difficulty"])

⏳ Researching Artificial Intelligence...

📝 SUMMARY:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI has numerous applications across various industries, including healthcare, finance, and transportation. The field of AI is rapidly evolving, with ongoing research focused on creating more advanced and autonomous systems.

🔑 KEY FACTS:
1. The term 'Artificial Intelligence' was coined in 1956 by John McCarthy
2. AI systems can be classified into two main categories: narrow or weak AI, and general or strong AI
3. Machine learning is a key subset of AI, enabling systems to learn from data and improve their performance over time
4. AI has the potential to significantly impact the job market, with some jobs becoming automated and others being created
5. The development of AI raises important ethical concerns, such as bias, privacy, and account

In [10]:
# Test if chain is working
test = research_chain.invoke({"topic": "Python"})
print(type(test))
print(test)

<class 'dict'>
{'summary': "Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, and data analysis. It is known for its simplicity, readability, and large community of developers who contribute to its ecosystem. Python's versatility and ease of use make it a popular choice among beginners and experts alike.", 'key_facts': ['Python was created in the late 1980s by Guido van Rossum', 'It is an object-oriented language that supports multiple programming paradigms', 'Python has a vast collection of libraries and frameworks, including NumPy, pandas, and Django', 'It is widely used in data science, machine learning, and artificial intelligence applications', 'Python is available on multiple platforms, including Windows, macOS, and Linux'], 'subtopics': ['Data Structures and Algorithms in Python', 'Web Development with Python Frameworks', 'Machine Learning and Artificial Intelligence with Python'], 'di

In [11]:
# User inputs their own topic
topic = input("🔍 Enter a topic you want to research: ")

print(f"\n⏳ Researching {topic}...")

result = research_chain.invoke({"topic": topic})

print("\n📝 SUMMARY:")
print(result["summary"])

print("\n🔑 KEY FACTS:")
for i, fact in enumerate(result["key_facts"], 1):
    print(f"{i}. {fact}")

print("\n🎯 SUBTOPICS TO EXPLORE:")
for subtopic in result["subtopics"]:
    print(f"• {subtopic}")

print("\n📊 DIFFICULTY:", result["difficulty"])


⏳ Researching education...

📝 SUMMARY:
Education is a vital aspect of human development, enabling individuals to acquire knowledge, skills, and values necessary for personal and professional growth. It encompasses various stages, from primary to higher education, and can be delivered through formal, non-formal, or informal settings. Effective education systems can have a profound impact on individuals, communities, and societies as a whole.

🔑 KEY FACTS:
1. The United Nations' Sustainable Development Goal 4 aims to ensure inclusive and equitable quality education for all by 2030.
2. Education can increase an individual's earning potential by up to 10% for each additional year of schooling.
3. The global literacy rate has increased from 81% in 2000 to 86% in 2019, with significant improvements in developing countries.
4. The use of technology in education, such as online learning platforms and digital resources, is becoming increasingly prevalent.
5. Access to quality education is a ma